# NextMove — Pickleball Detector: Transfer Learning on Kaggle GPU

This notebook fine-tunes a **COCO-pretrained YOLO11n** backbone on a public
pickleball detection dataset, then exports a **Core ML** model for the iOS app.

**Pipeline:** COCO-pretrained weights (transfer learning) → fine-tune on
pickleball data → validate → export to Core ML (`PickleballDetector_v1`).

> Enable GPU: Notebook settings → Accelerator → **GPU T4 x2** (free).

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow coremltools
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Download the public dataset (transfer-learning source)

Get a free Roboflow API key: https://app.roboflow.com/settings/api

We fine-tune on this openly published pickleball dataset — a standard,
reproducible starting point. Record its license (typically CC BY 4.0).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''  # <-- paste your free key

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('racket-ai').project('pickleball-iiv9m')
dataset = project.version(5).download('yolov8')
print('Dataset at:', dataset.location)
# Dataset classes: ball, paddle, player (nc=3). License: CC BY 4.0.
# ~12k train images. Training reads its own data.yaml automatically.

## 3. Transfer learning + fine-tuning

`YOLO('yolo11n.pt')` loads COCO-pretrained weights (**transfer learning**).
`.train(...)` then adapts all layers to pickleball (**fine-tuning**) at a low LR.

In [ ]:
from ultralytics import YOLO

# 1. Load COCO-pretrained backbone (transfer learning)
model = YOLO('yolo11n.pt')

# 2. Fine-tune on the pickleball dataset
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    fliplr=0.5,
    mosaic=1.0,
    device=0,
    project='runs/train',
    name='pickleball_detector',
)

## 4. Validate — report the REAL measured metrics

In [ ]:
best = YOLO('runs/train/pickleball_detector/weights/best.pt')
metrics = best.val(data=f'{dataset.location}/data.yaml')
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

## 5. Export to Core ML for the iOS app

In [ ]:
# imgsz=640 matches Ultralytics' Core ML mobile standard. nms=True bakes in
# non-max suppression so the Swift side stays simple.
best.export(format='coreml', nms=True, imgsz=640)
print('Core ML model exported. Rename to PickleballDetector_v1.mlpackage')
print('and drop into nextmove/Models/Pickleball/ in the Xcode project.')

## 6. Download the artifacts

Download from Kaggle's output panel:
- `runs/train/pickleball_detector/weights/best.pt` (PyTorch weights)
- the exported `.mlpackage` (Core ML model for iOS)
- `runs/train/pickleball_detector/` (training curves + confusion matrix for your report)